# Exploring FineWeb-Edu

**Description:** Stream a small sample from FineWeb-Edu and explore its text, quality scores, and source domains without downloading the full dataset.
**Level:** Beginner
**Tags:** Datasets, Hugging Face, NLP

[FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) is a large collection of educational web pages filtered from FineWeb. The complete dataset contains roughly 1.3 trillion tokens, so this notebook deliberately uses streaming and stops after a small number of records.

## Load a bounded streaming sample

The smallest official dataset configuration is `sample-10BT` (about 10 billion GPT-2 tokens). Its smallest Parquet shard is `013_00000.parquet`, listed at about 541 MB. Streaming lets us read only the data ranges needed for the first 200 records instead of downloading that file—or the full 28.5 GB sample.

> This cell needs an internet connection. Increase `SAMPLE_SIZE` carefully: streaming limits local downloads, but every record still consumes bandwidth and memory.

In [ ]:
from itertools import islice

from datasets import load_dataset

SHARD_URL = (
    "https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu/"
    "resolve/main/sample/10BT/013_00000.parquet"
)
SAMPLE_SIZE = 200

stream = load_dataset(
    "parquet",
    data_files={"train": SHARD_URL},
    split="train",
    streaming=True,
)
records = list(islice(stream, SAMPLE_SIZE))

print(f"Loaded {len(records)} streamed records.")

## Inspect the schema and one record

In [ ]:
print("Columns:", list(records[0]))
print("Source URL:", records[0]["url"])
print("Educational score:", records[0]["score"])
print("Token count:", records[0]["token_count"])
print("\nText preview:\n")
print(records[0]["text"][:1_000])

## Summarize document sizes

FineWeb-Edu includes a precomputed token count. We can compare it with the number of characters and words in our streamed sample.

In [ ]:
from statistics import mean, median

token_counts = [record["token_count"] for record in records]
word_counts = [len(record["text"].split()) for record in records]
character_counts = [len(record["text"]) for record in records]

summary = {
    "documents": len(records),
    "median_tokens": median(token_counts),
    "mean_tokens": round(mean(token_counts), 1),
    "median_words": median(word_counts),
    "median_characters": median(character_counts),
    "largest_document_tokens": max(token_counts),
}
summary

## Explore educational quality scores

The `score` field is the continuous classifier score; `int_score` is its rounded integer form. Higher values indicate content assessed as more educational.

In [ ]:
from collections import Counter

score_counts = Counter(record["int_score"] for record in records)
for score in sorted(score_counts):
    count = score_counts[score]
    bar = "█" * round(40 * count / len(records))
    print(f"Score {score}: {count:>3} {bar}")

## Find common source domains

In [ ]:
from urllib.parse import urlparse

domains = Counter(
    urlparse(record["url"]).netloc.removeprefix("www.")
    for record in records
)
domains.most_common(10)

## Preview the highest-scoring documents

A score is a useful signal, not a guarantee of correctness. Always inspect and clean web data for the needs of your project.

In [ ]:
top_records = sorted(records, key=lambda record: record["score"], reverse=True)[:3]

for index, record in enumerate(top_records, start=1):
    preview = " ".join(record["text"].split())[:300]
    print(f"{index}. score={record['score']:.2f} | {record['url']}")
    print(f"   {preview}…\n")

## Takeaways

- FineWeb-Edu is far too large for casual local download, but streaming makes bounded exploration practical.
- The dataset exposes text, provenance, language, size, and educational-quality fields.
- A small sequential sample is useful for learning the schema, but it is not necessarily representative of the entire dataset.
- For a more representative analysis, sample across several shards while keeping an explicit record limit.